In [ ]:
# Vision Transformer (ViT) for CIFAR-10 based on "An Image is Worth 16x16 Words"

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


# -------------------------------
# Patch Embedding
# -------------------------------
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels=3, emb_dim=256):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, emb_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # shape: [B, emb_dim, H/patch, W/patch]
        x = x.flatten(2).transpose(1, 2)  # shape: [B, N_patches, emb_dim]
        return x


# -------------------------------
# Multi-Head Self-Attention
# -------------------------------
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # shape: [B, heads, N, head_dim]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, D)
        return self.out(out)


# -------------------------------
# Transformer Encoder Block
# -------------------------------
class TransformerEncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadSelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


# -------------------------------
# Vision Transformer
# -------------------------------
class VisionTransformer(nn.Module):
    def __init__(self, img_size=32, patch_size=4, emb_dim=256, depth=6, heads=8, mlp_dim=512, n_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, emb_dim=emb_dim)
        num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, emb_dim))

        self.encoder = nn.Sequential(*[
            TransformerEncoderBlock(emb_dim, heads, mlp_dim) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, n_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        B, N, _ = x.shape
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embed[:, :N+1]
        x = self.encoder(x)
        x = self.norm(x[:, 0])  # Use CLS token
        return self.head(x)


# -------------------------------
# Training and Evaluation
# -------------------------------
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total




import itertools
# -------------------------------
# Run Hyperparameter Search
# -------------------------------
def run_hyperparameter_experiments():
    device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    trainloader = DataLoader(trainset, batch_size=256, shuffle=True, num_workers=4)
    testloader = DataLoader(testset, batch_size=8, shuffle=False, num_workers=4)

    emb_dims = [256,192]
    depths = [4, 6]
    heads = [4, 8]
    mlp_dims = [1024,768]

    best_acc = 0
    best_config = None

    for emb_dim, depth, head, mlp_dim in itertools.product(emb_dims, depths, heads, mlp_dims):
        print(f"\nRunning config: emb_dim={emb_dim}, depth={depth}, heads={head}, mlp_dim={mlp_dim}")
        config_name = f"emb{emb_dim}_depth{depth}_head{head}_mlp{mlp_dim}"
        model = VisionTransformer(
            patch_size=4,
            emb_dim=emb_dim,
            depth=depth,
            heads=head,
            mlp_dim=mlp_dim
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
        criterion = nn.CrossEntropyLoss()

        accs = []
        for epoch in range(50):
            train(model, trainloader, optimizer, criterion, device)
            acc = evaluate(model, testloader, device)
            accs.append(acc)
            print(f"Epoch {epoch+1}: Accuracy={acc:.4f}")


        # Save plot
        import os
        os.makedirs("plots", exist_ok=True)
        plt.figure()
        plt.plot(range(1, len(accs)+1), accs)
        plt.title(f"Accuracy over Epochs\n{config_name}")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(True)
        plt.savefig(f"plots/{config_name}.png")
        plt.close()

        final_acc = accs[-1]
        if final_acc > best_acc:
            best_acc = final_acc
            best_config = (emb_dim, depth, head, mlp_dim)

        if final_acc >= 0.80:
            print(f"\n Achieved >80% accuracy: {final_acc:.4f} with config: {best_config}")

    print("\nBest configuration:")
    print(f"Embedding Dim: {best_config[0]}, Depth: {best_config[1]}, Heads: {best_config[2]}, MLP Dim: {best_config[3]} => Accuracy: {best_acc:.4f}")


# Run hyperparameter search
if __name__ == '__main__':
    run_hyperparameter_experiments()

Files already downloaded and verified
Files already downloaded and verified

Running config: emb_dim=256, depth=4, heads=4, mlp_dim=1024
Epoch 1: Accuracy=0.4770
Epoch 2: Accuracy=0.5249
Epoch 3: Accuracy=0.5479
Epoch 4: Accuracy=0.5704
Epoch 5: Accuracy=0.5910
Epoch 6: Accuracy=0.5907
Epoch 7: Accuracy=0.6123


## click [Terminal output](CV_1.2.out) 
## click [All plots](plots)


### Best configuration:
Embedding Dim: 192, Depth: 6, Heads: 8, MLP Dim: 1024 => Accuracy: 0.6345

##  Best Configuration Without Augmentation

- **Embedding dimension:** 192  
- **Number of transformer layers:** 6  
- **Number of attention heads:** 8  
- **MLP hidden dimension:** 1024  
- **Test Accuracy:** 63.45%

---

##  Analysis

The hyperparameter exploration reveals several important insights:

- **Transformer Depth:**  
  Models with **6 layers** consistently outperformed those with 4 layers, suggesting that additional transformer layers help capture more complex relationships in the data.

- **Attention Heads:**  
  Configurations with **8 attention heads** generally performed better than those with 4, indicating that more attention heads allow the model to focus on different aspects of images simultaneously.

- **MLP Dimension:**  
  The larger **MLP dimension (1024)** typically yielded better results than the smaller one (768), showing that larger projection spaces benefit feature representation.

- **Embedding Dimension:**  
  Interestingly, the **smaller embedding dimension (192)** combined with more layers and heads performed well, suggesting a good balance between model capacity and potential overfitting.

---

The overall best configuration balances these factors effectively, with the experiment results showing that **model depth** and **attention mechanisms** are more critical for performance than embedding dimension size.
